# Notebook 2: Observabilidad con Langfuse

## Objetivos de aprendizaje
- Entender por qué la **observabilidad** es imprescindible para LLMs en producción
- Conocer el **modelo de datos** de Langfuse: **Trace -> Span -> Generation**
- Configurar Langfuse Cloud y verificar la conexión
- Instrumentar el agente TechShop con el decorador **`@observe`** (Langfuse SDK v3)
- Enriquecer trazas con **metadata** (user_id, session_id, context)
- Generar trazas, analizarlas en el dashboard **y por API**
- Entender las **4 categorías de métricas LLMOps**

## ¿Por qué este notebook?
En el Notebook 1 construimos un agente que "parece que funciona". Pero no podemos responder preguntas críticas: ¿usó las herramientas? ¿cuánto costó? ¿inventó información? **La observabilidad transforma la incertidumbre en datos accionables.**

> **Referencia principal:** [Langfuse Documentation](https://langfuse.com/docs) · [Langfuse Python SDK v3](https://langfuse.com/docs/sdk/python/decorators)

---

## Parte 0: Teoría — Observabilidad para LLMs

### El problema fundamental

En software clásico, un bug produce un **stacktrace explícito**:
```bash
ERROR: NullPointerException at UserService.java:42
```
Sabes exactamente **qué** falló, **dónde** y **por qué**.

Con LLMs, los fallos son **semánticos**:
```bash
User: "¿Tenéis portátiles para diseño gráfico?"
Agent: "Sí, te recomiendo el MacBook Pro M3 con pantalla Retina..."
-> TechShop NO vende MacBook. El agente INVENTÓ un producto.
-> No hay excepción. No hay error. Solo una respuesta plausible e incorrecta.
```

> **Sin observabilidad, no se puede distinguir entre una respuesta correcta y una alucinación convincente sin ver el flujo interno del agente.**

### ¿Qué es observabilidad para LLMs?

Observabilidad = poder responder estas preguntas **en cualquier momento** sobre **cualquier request**:

| Pregunta | Sin observabilidad | Con observabilidad |
|----------|-------------------|-------------------|
| ¿Qué query recibió? | No | Input capturado |
| ¿Qué herramientas llamó? | No | Spans por tool call |
| ¿Qué devolvieron las herramientas? | No | Tool results en spans |
| ¿Cuántos tokens consumió? | No | input_tokens + output_tokens |
| ¿Cuánto tardó? | No | Latencia desglosada |
| ¿Inventó información? | Solo leyendo manualmente | Compara tool result vs. respuesta |

> **Concepto:** La observabilidad no es lo mismo que el logging. Logging registra eventos individuales. Observabilidad te permite **reconstruir el flujo completo** de cada request y **entender por qué** se comportó como lo hizo.

### El modelo de datos: Trace -> Span -> Generation

Langfuse organiza la información en una **jerarquía de tres niveles**, inspirada en OpenTelemetry:

```mermaid
graph TD
    T["TRACE<br/>(una request completa del usuario)"]
    T --> S1["SPAN: input_guardrail<br/>duration: 5ms, result: safe"]
    T --> S2["SPAN: agent_processing"]
    T --> S3["SPAN: output_guardrail<br/>duration: 3ms, result: valid"]

    S2 --> G1["GENERATION: LLM call #1<br/>decidir herramienta<br/>tokens_in: 150, tokens_out: 30"]
    S2 --> S2a["SPAN: tool.search_catalog<br/>input: portátiles -> 2 productos"]
    S2 --> G2["GENERATION: LLM call #2<br/>generar respuesta<br/>tokens_in: 400, tokens_out: 200"]
```

| Concepto | Qué representa | Cuándo se crea |
|----------|---------------|----------------|
| **Trace** | Una request completa del usuario | Una por consulta al agente |
| **Span** | Una operación lógica dentro de la request | Cada función decorada con `@observe` |
| **Generation** | Una llamada específica al LLM | Cada invocación a la Converse API |

> **Referencia:** [Langfuse — Tracing](https://langfuse.com/docs/tracing) · [Langfuse — Data Model](https://langfuse.com/docs/tracing#trace)

### OpenTelemetry: el estándar emergente

Langfuse soporta [OpenTelemetry (OTEL)](https://opentelemetry.io/), el estándar de la industria para observabilidad. Los atributos semánticos `gen_ai.*` están siendo estandarizados para LLMs:

| Atributo OTEL | Significado | Ejemplo |
|---------------|------------|---------|
| `gen_ai.system` | Proveedor del modelo | `"aws.bedrock"` |
| `gen_ai.request.model` | Modelo solicitado | `"claude-haiku-4-5"` |
| `gen_ai.usage.input_tokens` | Tokens de entrada | `150` |
| `gen_ai.usage.output_tokens` | Tokens de salida | `200` |
| `gen_ai.response.finish_reason` | Razón de parada | `"end_turn"` |

> Strands Agents emite spans OpenTelemetry nativamente. Si configuras un endpoint OTEL, las trazas del agente se envían automáticamente — sin código adicional. Langfuse puede recibir estas trazas vía su [endpoint OTEL](https://langfuse.com/docs/integrations/opentelemetry).

> **Referencia:** [OpenTelemetry Semantic Conventions for GenAI](https://opentelemetry.io/docs/specs/semconv/gen-ai/)

### ¿Qué es Langfuse?

[**Langfuse**](https://langfuse.com) es una plataforma **open-source** de observabilidad y gestión para aplicaciones LLM. Ofrece:

| Capacidad | Descripción | Notebook del curso |
|-----------|------------|-------------------|
| **Tracing** | Captura el flujo completo de cada request | NB 2 (este) |
| **Prompt Management** | Versionado de prompts con labels y rollback | NB 3 |
| **Evaluations** | Datasets + scoring (manual, heurístico, LLM-as-judge) | NB 4 |
| **Dashboards** | Métricas agregadas (latencia, coste, calidad) | NB 2 (este) |

**¿Por qué Langfuse y no otra herramienta?**
- **Open source** — puedes self-hostear o usar el SaaS (Cloud)
- **Plan gratuito** suficiente para aprendizaje (50K observaciones/mes)
- **SDK Python nativo** con decorador `@observe` de una línea
- **Multi-provider** — funciona con Bedrock, OpenAI, Anthropic, etc.
- **Integración OTEL** — recibe trazas OpenTelemetry estándar

**Alternativas en el mercado:** LangSmith (LangChain), Arize Phoenix, Weights & Biases Weave, Datadog LLM Observability. Langfuse destaca por ser open-source y por su integración nativa con SDKs de Python.

> **Referencia:** [Langfuse — Why Langfuse](https://langfuse.com/docs) · [Langfuse — Self-hosting](https://langfuse.com/docs/deployment/self-host)

## Tutorial: Configurar Langfuse Cloud (5 minutos)

### Paso 1: Crear cuenta
1. Ve a **[cloud.langfuse.com](https://cloud.langfuse.com)**
2. Click en **Sign Up** (puedes usar GitHub, Google o email)
3. El plan **Hobby** es gratuito: 50K observaciones/mes, retención 30 días

### Paso 2: Crear proyecto
1. Una vez dentro, click en **New Project**
2. Nombre: `techshop-llmops`
3. Click **Create**

### Paso 3: Obtener API Keys
1. Ve a **Settings** en el menú lateral
2. Click en **API Keys**
3. Click en **Create API Key**
4. Copia los 3 valores:
   - `Public Key` -> `LANGFUSE_PUBLIC_KEY`
   - `Secret Key` -> `LANGFUSE_SECRET_KEY`
   - `Host` -> `https://cloud.langfuse.com`

### Paso 4: Actualizar tu .env
```
LANGFUSE_PUBLIC_KEY=pk-lf-xxxxxxxx
LANGFUSE_SECRET_KEY=sk-lf-xxxxxxxx
LANGFUSE_HOST=https://cloud.langfuse.com
```

> **Importante:** Nunca compartas tus Secret Keys. El archivo `.env` ya está en `.gitignore`.

---

## Parte 1: Setup y conexión a Langfuse

In [1]:
import strands
import langfuse
import boto3

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Verificar AWS
assert os.getenv("AWS_ACCESS_KEY_ID"), "Falta AWS_ACCESS_KEY_ID"

# Verificar Langfuse
assert os.getenv("LANGFUSE_PUBLIC_KEY"), "Falta LANGFUSE_PUBLIC_KEY - sigue el tutorial arriba"
assert os.getenv("LANGFUSE_SECRET_KEY"), "Falta LANGFUSE_SECRET_KEY"

print("[OK] Variables de entorno cargadas")

### 1.2 Conectar con Langfuse

El SDK de Langfuse se inicializa automáticamente leyendo las variables de entorno. Internamente, crea un **cliente HTTP** que envía las trazas al servidor de Langfuse de forma **asíncrona** (no bloquea tu código).

> **Referencia:** [Langfuse Python SDK — Initialization](https://langfuse.com/docs/sdk/python/low-level-sdk#initialize-client)

In [ ]:
from langfuse import Langfuse


# Langfuse lee automáticamente las variables de entorno
langfuse = Langfuse()


# Verificar conexión
try:
    langfuse.auth_check()
    host = os.getenv("LANGFUSE_HOST") or os.getenv("LANGFUSE_BASE_URL")
    print(f"[OK] Conectado a Langfuse: {host}")
except Exception as e:
    print(f"[ERROR] Error de conexión: {e}")
    print("   Verifica LANGFUSE_PUBLIC_KEY y LANGFUSE_SECRET_KEY en tu .env")

**¿Qué hace `Langfuse()` internamente?**

1. Lee `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` y `LANGFUSE_HOST` del entorno
2. Crea un **buffer en memoria** donde se acumulan las observaciones
3. Envía las trazas al servidor en **batches** (cada pocos segundos o con `flush()`)
4. `auth_check()` hace un ping al servidor para verificar que las credenciales son válidas

> **Importante en notebooks:** En scripts y servidores, Langfuse envía trazas automáticamente. En notebooks, necesitamos llamar a `langfuse.flush()` explícitamente para forzar el envío antes de ir al dashboard.

---

## Parte 2: Tu primera traza con `@observe`

### El decorador `@observe` — la pieza central

El decorador [`@observe`](https://langfuse.com/docs/sdk/python/decorators) de Langfuse v3 es la forma más directa de instrumentar código Python. Lo que hace automáticamente:

| Captura automática | Descripción |
|-------------------|-------------|
| **Input** | Los argumentos de la función |
| **Output** | El valor de retorno |
| **Duración** | Timestamp inicio -> fin |
| **Excepciones** | Si la función lanza una excepción, se registra |
| **Jerarquía** | Si una función `@observe` llama a otra `@observe`, se crea un árbol padre->hijo |

```mermaid
graph TD
    T["Trace (se crea para la función raíz)"]
    T --> S["Span (@observe crea un span)"]
    S --> SS["Sub-span (función @observe llamada dentro)"]
```

### Parámetro `name` — siempre obligatorio en nuestro proyecto

```python
@observe(name="descriptive_name")  # Buena práctica
@observe()                          # Nombre auto-generado, difícil de filtrar
```

Convención de nombres: `<categoría>_<acción>` -> `tool_search_catalog`, `scan_input`, `agent_process_query`

> **Referencia:** [Langfuse — Python Decorators (@observe)](https://langfuse.com/docs/sdk/python/decorators)

In [ ]:
from langfuse import observe
import time


@observe(name="mi_primera_funcion")
def mi_funcion(x: int) -> int:
    """Una función simple instrumentada con Langfuse."""
    time.sleep(0.1)  # Simular trabajo
    return x * 2

# Ejecutar
resultado = mi_funcion(21)
print(f"Resultado: {resultado}")

# IMPORTANTE: en notebooks, hacemos flush manual para enviar las trazas
langfuse.flush()
print("\n-> Ve a Langfuse > Traces y busca 'mi_primera_funcion'")
print("  Verás: timestamp, duración, input (21), output (42)")

**¿Qué acaba de pasar?**

1. `@observe(name="mi_primera_funcion")` interceptó la llamada a `mi_funcion(21)`
2. Langfuse registró: input=`21`, output=`42`, duración ≈ 100ms
3. Creó un **trace** (porque es la función raíz) con un **span** dentro
4. La información está en el buffer local — `langfuse.flush()` la envió al servidor

Ve a **Langfuse -> Traces** y busca el trace con nombre `mi_primera_funcion`. Verás el timestamp, duración, input y output.

> **Nótese:** Con una sola línea de código (`@observe`) obtuvimos trazabilidad completa. Este es el valor de un SDK de observabilidad bien diseñado.

### Trazas anidadas — jerarquía automática

Cuando una función `@observe` llama a otra `@observe`, Langfuse crea automáticamente una **jerarquía padre->hijo**. Esto es lo que nos permite ver el árbol completo de operaciones de un agente.

In [ ]:
@observe(name="paso_validar")
def validar(texto: str) -> bool:
    """Simula una validación de input."""
    time.sleep(0.05)
    return len(texto) > 0


@observe(name="paso_procesar")
def procesar(texto: str) -> str:
    """Simula procesamiento."""
    time.sleep(0.1)
    return texto.upper()


@observe(name="pipeline_completo")
def pipeline(texto: str) -> str:
    """Pipeline que orquesta validación + procesamiento.
    
    En Langfuse verás la jerarquía:
    pipeline_completo (trace)
    -> paso_validar (span hijo)
    -> paso_procesar (span hijo)
    """
    if validar(texto):
        return procesar(texto)
    return "Input inválido"


resultado = pipeline("hola mundo")
print(f"Resultado: {resultado}")

langfuse.flush()
print("\n-> Ve a Langfuse > Traces > 'pipeline_completo'")
print("  Verás el árbol: pipeline_completo -> paso_validar + paso_procesar")

---

## Parte 3: Instrumentar el agente TechShop

### 3.1 Importar el agente empaquetado

A partir de ahora usamos el agente de `src/techshop_agent/` — el mismo que construimos en el NB 1, pero empaquetado con búsqueda robusta y normalización Unicode.

In [ ]:
from techshop_agent import create_agent

agent = create_agent()

print("[OK] Agente TechShop importado desde el paquete")

### 3.2 Instrumentar con `@observe` + metadata

Crear una función wrapper que envuelva la llamada al agente con trazabilidad completa. Además de la captura automática de `@observe`, añadimos **metadata** con [`langfuse_context`](https://langfuse.com/docs/sdk/python/decorators#set-trace-params):

| Metadata | Para qué sirve en producción |
|----------|------------------------------|
| `user_id` | Agrupar queries por usuario, detectar patrones individuales |
| `session_id` | Reconstruir conversaciones completas en orden |
| `metadata.source` | Saber de dónde vino la query (web, API, notebook) |
| `metadata.query_length` | Correlacionar longitud de query con calidad de respuesta |

> **Referencia:** [Langfuse — Update Trace/Span](https://langfuse.com/docs/sdk/python/decorators#set-trace-params)

In [ ]:
# Actualizar metadata de la traza
from langfuse import langfuse_context


@observe(name="techshop_query")
def process_query(user_query: str, user_id: str = "student", session_id: str = "nb02") -> str:
    """Procesa una consulta al agente con trazabilidad Langfuse.

    Langfuse captura automáticamente:
    - Timestamp de inicio y fin
    - Input (user_query) y Output (respuesta)
    - Duración total
    """
    langfuse_context.update_current_trace(
        user_id=user_id,
        session_id=session_id,
        metadata={"source": "notebook_02", "query_length": len(user_query)},
    )

    response = agent(user_query)
    return str(response)

print("[OK] Función process_query instrumentada con Langfuse")

**Análisis del código de instrumentación:**

1. **`@observe(name="techshop_query")`** — Crea un trace para cada llamada. El `name` nos permite filtrar en el dashboard.
2. **`langfuse_context.update_current_trace(...)`** — Enriquece el trace con metadata. Es como añadir "etiquetas" que luego usaremos para filtrar y agrupar.
3. **`user_id` y `session_id`** — Son ciudadanos de primera clase en Langfuse: tienen filtros dedicados en el dashboard.
4. **`metadata`** — Es un dict libre donde puedes poner cualquier información contextual.

> **Convención del proyecto:** En `src/techshop_agent/`, cada función que interactúa con el LLM o con herramientas lleva `@observe`. Los nombres siguen el patrón `<categoría>_<acción>` (ej: `tool_search_catalog`, `scan_input`).

### 3.3 Generar trazas con consultas diversas

Ejecutamos **8 consultas variadas** para generar un conjunto representativo de trazas. La variedad es intencionada:

| Queries 1-2 | Queries 3-4 | Queries 5-6 | Queries 7-8 |
|-------------|-------------|-------------|-------------|
| Productos directos | FAQs | Fuera de ámbito + ambigua | Combinadas |

Esto nos permitirá observar en el dashboard diferencias de **latencia**, **uso de herramientas** y **patrones de respuesta**.

In [ ]:
queries = [
    "¿Qué portátiles tenéis disponibles?",
    "¿Cuál es la política de devoluciones?",
    "Quiero un móvil con buena cámara",
    "¿Hacéis envíos internacionales?",
    "¿Cuál es la capital de Francia?",
    "Necesito unos auriculares con cancelación de ruido",
    "¿Tenéis tablets?",
    "¿Puedo pagar en cuotas?",
]

for i, query in enumerate(queries, 1):
    print(f"\n{'=' * 60}")
    print(f"Query {i}/{len(queries)}: {query}")
    print('=' * 60)
    try:
        response = process_query(
            user_query=query,
            user_id="student01",
            session_id="session-nb02-demo",
        )
        # Mostrar solo primeros 200 chars para no saturar el output
        preview = response[:200] + "..." if len(response) > 200 else response
        print(preview)
    except Exception as e:
        print(f"[ERROR] {e}")

# Flush para enviar todas las trazas
langfuse.flush()
print(f"\n\n[OK] {len(queries)} trazas enviadas a Langfuse")

---

## Parte 4: Analizar trazas en el dashboard

Ve a **[cloud.langfuse.com](https://cloud.langfuse.com)** -> tu proyecto -> **Traces**.

### Guía de exploración paso a paso

**1. Vista de Traces (lista)**
- Cada fila = una consulta del usuario
- Columnas: nombre, duración, user_id, timestamp
- Filtra por `user_id = student01` para ver solo tus trazas

**2. Vista detallada (click en una traza)**
- Verás el **árbol de spans**: qué operaciones se ejecutaron
- Input del usuario (la query original)
- Output del agente (la respuesta generada)
- Metadata que añadimos (`source`, `query_length`)

**3. Preguntas de análisis**

| Pregunta | Dónde buscar | Qué implica |
|----------|-------------|-------------|
| ¿Hay queries donde el agente NO usó herramientas? | Árbol de spans (traces sin tool calls) | Posible alucinación |
| ¿Alguna query tardó mucho más? | Columna Duration | Cuello de botella |
| ¿Respondió preguntas fuera de ámbito? | Output de la query "capital de Francia" | Falta de guardrails |
| ¿El session_id agrupa bien? | Filtro Session | Reconstrucción de conversaciones |

> **Referencia:** [Langfuse — Traces UI](https://langfuse.com/docs/tracing)

---

## Parte 5: Consultar trazas por API

El dashboard es útil para exploración manual, pero en LLMOps necesitamos **análisis programático**. El [SDK de Langfuse](https://langfuse.com/docs/sdk/python/low-level-sdk) permite consultar trazas, observaciones y sesiones por API.

### Métodos clave del cliente `Langfuse`

| Método | Qué devuelve | Uso típico |
|--------|-------------|-----------|
| `fetch_traces(limit=N)` | Lista de traces con metadata | Análisis de latencia, errores |
| `fetch_trace(id)` | Un trace completo con sus observaciones | Debug de una request específica |
| `fetch_observations(trace_id=id)` | Spans y generations de un trace | Ver qué herramientas se llamaron |
| `fetch_sessions()` | Sesiones agrupadas | Reconstruir conversaciones |

> **Referencia:** [Langfuse Python SDK — Fetch data](https://langfuse.com/docs/sdk/python/low-level-sdk#fetch-traces)

In [ ]:
# Obtener las últimas trazas
traces = langfuse.fetch_traces(limit=10)

print(f"Últimas {len(traces.data)} trazas:\n")
for t in traces.data:
    duration_ms = "N/A"
    if t.latency is not None:
        duration_ms = f"{t.latency * 1000:.0f}ms"

    input_preview = ""
    if t.input:
        input_text = str(t.input)
        input_preview = input_text[:50] + "..." if len(input_text) > 50 else input_text

    print(f"  [{t.name or 'unnamed':20s}] duration={duration_ms:>8s} | user={t.user_id or 'N/A'} | {input_preview}")

In [ ]:
# Análisis más profundo: ver las observaciones (spans) de un trace específico
if traces.data:
    first_trace = traces.data[0]
    print(f"Detalle del trace: {first_trace.name} (id: {first_trace.id[:12]}...)")
    print(f"   User: {first_trace.user_id}")
    print(f"   Session: {first_trace.session_id}")
    print(f"   Latency: {first_trace.latency * 1000:.0f}ms" if first_trace.latency else "   Latency: N/A")
    
    # Obtener observaciones (spans hijos) de este trace
    observations = langfuse.fetch_observations(trace_id=first_trace.id)
    print(f"\n   Observaciones ({len(observations.data)} spans):")
    for obs in observations.data:
        obs_type = obs.type or "span"
        duration = f"{obs.latency * 1000:.0f}ms" if obs.latency else "N/A"
        print(f"      [{obs_type:12s}] {obs.name or 'unnamed':25s} | duration={duration}")
else:
    print("No se encontraron trazas. Ejecuta la celda de queries primero.")

---

## Parte 6: Métricas clave de observabilidad LLMOps

La observabilidad no es solo "ver trazas". Es poder responder preguntas de negocio con datos.

### Las 4 categorías de métricas LLMOps

| Categoría | Qué medimos | Ejemplos | Herramientas |
|-----------|-------------|----------|-------------|
| **Operacionales** | Disponibilidad, latencia, errores | P50 latencia=2.3s, error rate=0.5% | Langfuse, CloudWatch, Datadog |
| **Coste** | Tokens consumidos, € por request | $0.003/query, 500 input + 200 output tokens | Langfuse (tokens), billing APIs |
| **Calidad** | Alucinaciones, relevancia, fidelidad | 15% queries sin tool call, ROUGE=0.8 | Evaluaciones (NB 4), LLM-as-judge |
| **Uso** | Queries/hora, temas, patrones | 60% productos, 30% FAQs, 10% fuera de ámbito | Langfuse analytics, dashboards |

### ¿Qué mide cada herramienta?

```mermaid
flowchart TD
    subgraph Infra["Métricas Operacionales"]
        CW["CloudWatch / Datadog<br/>Latencia P50/P99<br/>Error rate<br/>Throughput"]
    end
    subgraph Calidad["Métricas de Calidad"]
        EV["Evaluaciones LLMOps<br/>Alucinaciones<br/>Relevancia<br/>Fidelidad"]
    end
    subgraph Puente["Langfuse — Puente entre infra y calidad"]
        LF["Trazas · Tokens/coste<br/>Prompts · Sesiones"]
    end

    CW --> LF
    EV --> LF
```

> **Insight clave:** Las categorías 1 y 2 (operacionales, coste) se pueden medir con herramientas de infraestructura clásicas. Las categorías 3 y 4 (calidad, uso) **necesitan herramientas LLMOps** — por eso Langfuse es tan importante.

---

## Ejercicios opcionales

### Ejercicio 1: Añade metadata personalizado

Modifica `process_query` para capturar la **longitud de la respuesta** y el **número de palabras** como metadata adicional. Luego consulta las trazas por API y analiza la correlación entre longitud de query y longitud de respuesta.

> **Pista:** Usa `langfuse_context.update_current_observation()` para añadir metadata al span actual.

In [ ]:
# === EJERCICIO 1: Metadata enriquecido ===
# Descomenta y modifica para añadir más metadata

# from langfuse import observe, langfuse_context
#
# @observe(name="techshop_query_enriched")
# def process_query_enriched(user_query: str, user_id: str = "student", session_id: str = "nb02") -> str:
#     langfuse_context.update_current_trace(
#         user_id=user_id,
#         session_id=session_id,
#         metadata={"source": "notebook_02", "query_length": len(user_query)},
#     )
#
#     response = agent(user_query)
#     response_str = str(response)
#
#     # TODO: Añade metadata sobre la respuesta
#     langfuse_context.update_current_observation(
#         metadata={
#             "response_length": len(response_str),
#             "response_word_count": len(response_str.split()),
#         }
#     )
#
#     return response_str
#
# result = process_query_enriched("¿Qué portátiles tenéis?")
# print(result[:200])
# langfuse.flush()

### Ejercicio 2: Análisis de latencia por API

Usa la API de Langfuse para calcular la **latencia media** y la **latencia P90** de tus trazas. ¿Hay queries que tardan significativamente más? ¿Se correlaciona con el tipo de query (producto vs FAQ vs fuera de ámbito)?

In [ ]:
# === EJERCICIO 2: Análisis de latencia ===
# Descomenta y ejecuta

# import statistics
#
# traces_data = langfuse.fetch_traces(limit=20)
# latencies = [t.latency * 1000 for t in traces_data.data if t.latency is not None]
#
# if latencies:
#     print(f"Análisis de latencia ({len(latencies)} trazas):")
#     print(f"   Media:    {statistics.mean(latencies):.0f} ms")
#     print(f"   Mediana:  {statistics.median(latencies):.0f} ms")
#     print(f"   Min:      {min(latencies):.0f} ms")
#     print(f"   Max:      {max(latencies):.0f} ms")
#     if len(latencies) >= 2:
#         print(f"   Desv.Std: {statistics.stdev(latencies):.0f} ms")
#     # P90 aproximado
#     sorted_lat = sorted(latencies)
#     p90_idx = int(len(sorted_lat) * 0.9)
#     print(f"   P90:      {sorted_lat[p90_idx]:.0f} ms")
# else:
#     print("No hay latencias disponibles. Ejecuta las queries primero.")

### Ejercicio 3: Scoring manual de trazas

Langfuse permite [asignar scores](https://langfuse.com/docs/scores/custom) a las trazas, tanto manualmente como programáticamente. Esto es la base de las evaluaciones que veremos en el NB 4.

Prueba a asignar un score a una traza indicando si la respuesta fue correcta o no:

In [ ]:
# === EJERCICIO 3: Scoring de trazas ===
# Descomenta y ejecuta. Luego ve al dashboard y filtra por score.

# if traces.data:
#     trace_to_score = traces.data[0]
#
#     # Score numérico (0-1) — ¿la respuesta fue correcta?
#     langfuse.score(
#         trace_id=trace_to_score.id,
#         name="correctness",
#         value=1.0,  # 1.0 = correcta, 0.0 = incorrecta
#         comment="Respuesta verificada manualmente: usó la herramienta correcta"
#     )
#
#     # Score categórico — ¿qué tipo de query fue?
#     langfuse.score(
#         trace_id=trace_to_score.id,
#         name="query_type",
#         value="product_search",
#         comment="Query sobre productos del catálogo"
#     )
#
#     langfuse.flush()
#     print(f"[OK] Scores asignados al trace {trace_to_score.id[:12]}...")
#     print("   -> Ve a Langfuse > Traces > click en el trace > pestaña Scores")
# else:
#     print("Ejecuta las queries primero")

---

## Resumen y conceptos clave

### Antes vs. después

| Sin observabilidad (NB 1) | Con observabilidad (NB 2) |
|---------------------------|---------------------------|
| "El agente respondió algo" | Veo exactamente qué queries recibió |
| "Parece que funciona" | Conozco la latencia de cada llamada |
| "No sé si usó las herramientas" | Veo el árbol completo de spans |
| "¿Cuánto cuesta?" | Puedo ver tokens consumidos por request |

### Conceptos que debes llevarte

- **Trace -> Span -> Generation**: la jerarquía que estructura toda la observabilidad
- **`@observe`**: una línea de código = trazabilidad completa (input, output, duración, errores)
- **`langfuse_context`**: para enriquecer trazas con metadata contextual (user_id, session_id)
- **`langfuse.flush()`**: en notebooks, siempre flush antes de ir al dashboard
- **API programática**: `fetch_traces()`, `fetch_observations()`, `score()` para análisis automatizado
- **4 categorías de métricas**: operacionales, coste, calidad, uso — las dos últimas necesitan LLMOps

### Mapa del curso

| Notebook | Tema | Estado |
|----------|------|--------|
| NB 1 | Construir el agente | Completado |
| **NB 2** | **Observar qué hace (trazas)** | **Actual** |
| NB 3 | Versionar y gestionar prompts (Langfuse Prompt Management) | Siguiente |
| NB 4 | Evaluar si lo hace bien (LLM-as-Judge, datasets) | Pendiente |
| NB 5 | CI/CD Gate automatizado | Pendiente |
| NB 6 | Proteger input y output (guardrails) | Pendiente |
| NB 7 | Pipeline integrado (todo junto) | Pendiente |
| NB 7 | Pipeline integrado (todo junto) | Pendiente |

### Referencias del notebook

- [Langfuse — Documentation](https://langfuse.com/docs)
- [Langfuse — Python SDK Decorators (@observe)](https://langfuse.com/docs/sdk/python/decorators)
- [Langfuse — Tracing Data Model](https://langfuse.com/docs/tracing)
- [Langfuse — Python SDK Low-level (fetch, score)](https://langfuse.com/docs/sdk/python/low-level-sdk)
- [Langfuse — Scores](https://langfuse.com/docs/scores/custom)
- [OpenTelemetry — Semantic Conventions for GenAI](https://opentelemetry.io/docs/specs/semconv/gen-ai/)
- [Strands Agents — Observability](https://strandsagents.com/latest/user-guide/concepts/agents/)

---

## Siguiente: [Notebook 3 — Prompt Management con Langfuse](./03_prompt_management.ipynb)